# Disparity Analysis Audit Program

This program assesses disparate impact under equalized odds framework by comparing
coverage and annotation quality metrics across target groups within the same coding level.

In [ ]:
# Imports
from dataclasses import dataclass
from pathlib import Path

import pandas as pd


In [ ]:
@dataclass
class DisparityAuditConfig:
    # `workdir` is the anchor used to resolve all relative paths.
    workdir: Path

    # Stage-01 outputs consumed by this notebook.
    annotation_output_dir: Path = Path('../outputs/annotation_audits')

    # Directory where disparity exports are written.
    output_dir: Path = Path('../outputs/disparity_audits')

In [ ]:
# Configuration
WORKDIR = Path.cwd()
cfg = DisparityAuditConfig(workdir=WORKDIR)

# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.annotation_output_path = cfg.workdir / cfg.annotation_output_dir
cfg.coverage_report_path = cfg.annotation_output_path / 'coverage_metrics_upstream.tsv'
cfg.annotation_quality_path = cfg.annotation_output_path / 'annotation_quality.tsv'

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

cfg

In [ ]:
# Load stage-01 pipeline outputs
coverage_report = pd.read_csv(cfg.coverage_report_path, sep='\t')
annotation_quality_report = pd.read_csv(cfg.annotation_quality_path, sep='\t')

print(f"Loaded upstream coverage rows: {len(coverage_report)}")
print(f"Loaded upstream annotation rows: {len(annotation_quality_report)}")

In [ ]:
# Validate upstream schema for coverage metrics
required_coverage_cols = {
    'taxonomy_level', 'target', 'presence_rate', 'type_coverage', 'token_frequency'
}
missing_coverage_cols = required_coverage_cols - set(coverage_report.columns)
if missing_coverage_cols:
    raise ValueError(
        f"Missing required columns in upstream coverage report: {sorted(missing_coverage_cols)}"
    )

coverage_report.head()

In [ ]:
# Validate upstream schema for annotation quality metrics
required_annotation_cols = {
    'taxonomy_level', 'target',
    'correct_labeling_rate', 'annotator_failure_ratio',
    'case_a_present_hateful', 'case_b_present_nonhateful'
}
missing_annotation_cols = required_annotation_cols - set(annotation_quality_report.columns)
if missing_annotation_cols:
    raise ValueError(
        f"Missing required columns in upstream annotation report: {sorted(missing_annotation_cols)}"
    )

annotation_quality_report.head()

In [ ]:
# Ensure one row per (taxonomy_level, target) for pairwise disparities
coverage_dup_keys = coverage_report.duplicated(['taxonomy_level', 'target'])
annotation_dup_keys = annotation_quality_report.duplicated(['taxonomy_level', 'target'])
if coverage_dup_keys.any():
    raise ValueError('Upstream coverage report has duplicate (taxonomy_level, target) rows.')
if annotation_dup_keys.any():
    raise ValueError('Upstream annotation report has duplicate (taxonomy_level, target) rows.')

print('Upstream reports passed validation checks.')

In [ ]:
# Upstream reports are now the direct inputs for disparity analysis
print("Coverage Report:")
print(coverage_report)
print("\nAnnotation Quality Report:")
print(annotation_quality_report)

In [ ]:
# Coverage Disparity Analysis: Compare metrics across target groups within same level
coverage_disparities = []

for level in coverage_report['taxonomy_level'].unique():
    level_data = coverage_report[coverage_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Presence rate disparity
                presence_gap = data_a['presence_rate'] - data_b['presence_rate']
                presence_gap_abs = abs(presence_gap)

                # Presence rate disparate impact ratio (80% rule: < 0.8 flags concern)
                presence_max = max(data_a['presence_rate'], data_b['presence_rate'])
                presence_di_ratio = (
                    min(data_a['presence_rate'], data_b['presence_rate']) / presence_max
                    if presence_max > 0 else 1.0
                )

                # Type coverage disparity
                type_gap = data_a['type_coverage'] - data_b['type_coverage']
                type_gap_abs = abs(type_gap)

                # Type coverage disparate impact ratio
                type_max = max(data_a['type_coverage'], data_b['type_coverage'])
                type_di_ratio = (
                    min(data_a['type_coverage'], data_b['type_coverage']) / type_max
                    if type_max > 0 else 1.0
                )

                # Token frequency disparity (relative)
                if data_b['token_frequency'] > 0:
                    token_ratio = data_a['token_frequency'] / data_b['token_frequency']
                else:
                    token_ratio = float('inf') if data_a['token_frequency'] > 0 else 1.0

                coverage_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'presence_rate_a': data_a['presence_rate'],
                    'presence_rate_b': data_b['presence_rate'],
                    'presence_rate_gap': presence_gap,
                    'presence_rate_gap_abs': presence_gap_abs,
                    'presence_rate_di_ratio': presence_di_ratio,
                    'type_coverage_a': data_a['type_coverage'],
                    'type_coverage_b': data_b['type_coverage'],
                    'type_coverage_gap': type_gap,
                    'type_coverage_gap_abs': type_gap_abs,
                    'type_coverage_di_ratio': type_di_ratio,
                    'token_freq_a': data_a['token_frequency'],
                    'token_freq_b': data_b['token_frequency'],
                    'token_freq_ratio': token_ratio
                })

coverage_disparity_report = pd.DataFrame(coverage_disparities)


In [ ]:
# Annotation Quality Disparity Analysis: Compare labeling metrics across target groups within same level
annotation_disparities = []

for level in annotation_quality_report['taxonomy_level'].unique():
    level_data = annotation_quality_report[annotation_quality_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Correct labeling rate disparity
                labeling_gap = data_a['correct_labeling_rate'] - data_b['correct_labeling_rate']
                labeling_gap_abs = abs(labeling_gap)

                # Annotator failure ratio disparity
                failure_gap = data_a['annotator_failure_ratio'] - data_b['annotator_failure_ratio']
                failure_gap_abs = abs(failure_gap)

                annotation_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'correct_labeling_rate_a': data_a['correct_labeling_rate'],
                    'correct_labeling_rate_b': data_b['correct_labeling_rate'],
                    'labeling_rate_gap': labeling_gap,
                    'labeling_rate_gap_abs': labeling_gap_abs,
                    'annotator_failure_ratio_a': data_a['annotator_failure_ratio'],
                    'annotator_failure_ratio_b': data_b['annotator_failure_ratio'],
                    'failure_ratio_gap': failure_gap,
                    'failure_ratio_gap_abs': failure_gap_abs,
                    'case_a_a': data_a['case_a_present_hateful'],
                    'case_a_b': data_b['case_a_present_hateful'],
                    'case_b_a': data_a['case_b_present_nonhateful'],
                    'case_b_b': data_b['case_b_present_nonhateful']
                })

annotation_disparity_report = pd.DataFrame(annotation_disparities)

In [ ]:
# Cross-level consistency: for each target group, track how metrics change across taxonomy levels.
# Systematic degradation at higher levels may indicate a structural collection or annotation gap.
cross_level_data = []

all_targets = set(coverage_report['target'].unique()) | set(annotation_quality_report['target'].unique())

for target in sorted(all_targets):
    coverage_target = coverage_report[coverage_report['target'] == target].sort_values('taxonomy_level')
    annotation_target = annotation_quality_report[annotation_quality_report['target'] == target].sort_values('taxonomy_level')

    all_levels = sorted(set(coverage_target['taxonomy_level'].tolist()) | set(annotation_target['taxonomy_level'].tolist()))

    if len(all_levels) > 1:
        for k in range(len(all_levels) - 1):
            level_lo = all_levels[k]
            level_hi = all_levels[k + 1]

            # Coverage deltas
            cov_lo = coverage_target[coverage_target['taxonomy_level'] == level_lo]
            cov_hi = coverage_target[coverage_target['taxonomy_level'] == level_hi]
            presence_delta = (
                cov_hi.iloc[0]['presence_rate'] - cov_lo.iloc[0]['presence_rate']
                if len(cov_lo) > 0 and len(cov_hi) > 0 else None
            )
            type_delta = (
                cov_hi.iloc[0]['type_coverage'] - cov_lo.iloc[0]['type_coverage']
                if len(cov_lo) > 0 and len(cov_hi) > 0 else None
            )

            # Annotation deltas
            ann_lo = annotation_target[annotation_target['taxonomy_level'] == level_lo]
            ann_hi = annotation_target[annotation_target['taxonomy_level'] == level_hi]
            labeling_delta = (
                ann_hi.iloc[0]['correct_labeling_rate'] - ann_lo.iloc[0]['correct_labeling_rate']
                if len(ann_lo) > 0 and len(ann_hi) > 0 else None
            )
            failure_delta = (
                ann_hi.iloc[0]['annotator_failure_ratio'] - ann_lo.iloc[0]['annotator_failure_ratio']
                if len(ann_lo) > 0 and len(ann_hi) > 0 else None
            )

            cross_level_data.append({
                'target': target,
                'level_from': level_lo,
                'level_to': level_hi,
                'presence_rate_from': cov_lo.iloc[0]['presence_rate'] if len(cov_lo) > 0 else None,
                'presence_rate_to': cov_hi.iloc[0]['presence_rate'] if len(cov_hi) > 0 else None,
                'presence_rate_delta': presence_delta,
                'type_coverage_from': cov_lo.iloc[0]['type_coverage'] if len(cov_lo) > 0 else None,
                'type_coverage_to': cov_hi.iloc[0]['type_coverage'] if len(cov_hi) > 0 else None,
                'type_coverage_delta': type_delta,
                'correct_labeling_rate_from': ann_lo.iloc[0]['correct_labeling_rate'] if len(ann_lo) > 0 else None,
                'correct_labeling_rate_to': ann_hi.iloc[0]['correct_labeling_rate'] if len(ann_hi) > 0 else None,
                'correct_labeling_rate_delta': labeling_delta,
                'annotator_failure_ratio_from': ann_lo.iloc[0]['annotator_failure_ratio'] if len(ann_lo) > 0 else None,
                'annotator_failure_ratio_to': ann_hi.iloc[0]['annotator_failure_ratio'] if len(ann_hi) > 0 else None,
                'annotator_failure_ratio_delta': failure_delta,
            })

cross_level_report = pd.DataFrame(cross_level_data)


In [ ]:
# Print results
print("=" * 120)
print("DISPARITY ANALYSIS AUDIT")
print("=" * 120)

if len(coverage_disparity_report) > 0:
    print("\n" + "=" * 120)
    print("COVERAGE DISPARITY ANALYSIS: Within-Level Comparisons")
    print("=" * 120)
    print("\nPresence Rate Disparities (gap > 0 = target_a has better coverage; DI ratio < 0.8 flags concern):")
    presence_cols = ['taxonomy_level', 'target_a', 'target_b', 'presence_rate_a', 'presence_rate_b', 'presence_rate_gap', 'presence_rate_gap_abs', 'presence_rate_di_ratio']
    print(coverage_disparity_report[presence_cols].to_string(index=False))

    print("\nType Coverage Disparities (gap > 0 = target_a has better type representation; DI ratio < 0.8 flags concern):")
    type_cols = ['taxonomy_level', 'target_a', 'target_b', 'type_coverage_a', 'type_coverage_b', 'type_coverage_gap', 'type_coverage_gap_abs', 'type_coverage_di_ratio']
    print(coverage_disparity_report[type_cols].to_string(index=False))

    print("\nToken Frequency Ratios (ratio > 1 = target_a appears more frequently):")
    token_cols = ['taxonomy_level', 'target_a', 'target_b', 'token_freq_a', 'token_freq_b', 'token_freq_ratio']
    print(coverage_disparity_report[token_cols].to_string(index=False))

if len(annotation_disparity_report) > 0:
    print("\n" + "=" * 120)
    print("ANNOTATION QUALITY DISPARITY ANALYSIS: Within-Level Comparisons")
    print("=" * 120)
    print("\nCorrect Labeling Rate Disparities (higher = target_a has better annotation quality):")
    labeling_cols = ['taxonomy_level', 'target_a', 'target_b', 'correct_labeling_rate_a', 'correct_labeling_rate_b', 'labeling_rate_gap', 'labeling_rate_gap_abs']
    print(annotation_disparity_report[labeling_cols].to_string(index=False))

    print("\nAnnotator Failure Ratio Disparities (higher = target_a has worse annotation quality):")
    failure_cols = ['taxonomy_level', 'target_a', 'target_b', 'annotator_failure_ratio_a', 'annotator_failure_ratio_b', 'failure_ratio_gap', 'failure_ratio_gap_abs']
    print(annotation_disparity_report[failure_cols].to_string(index=False))

    print("\nCase Counts for Context:")
    case_cols = ['taxonomy_level', 'target_a', 'target_b', 'case_a_a', 'case_b_a', 'case_a_b', 'case_b_b']
    print(annotation_disparity_report[case_cols].to_string(index=False))

if len(cross_level_report) > 0:
    print("\n" + "=" * 120)
    print("CROSS-LEVEL CONSISTENCY: Within-Group Coverage Trends Across Taxonomy Levels")
    print("=" * 120)
    print("\nNegative delta = coverage degrades at higher specificity levels.")
    coverage_cross_cols = ['target', 'level_from', 'level_to', 'presence_rate_from', 'presence_rate_to', 'presence_rate_delta', 'type_coverage_from', 'type_coverage_to', 'type_coverage_delta']
    print(cross_level_report[coverage_cross_cols].to_string(index=False))

    print("\n" + "=" * 120)
    print("CROSS-LEVEL CONSISTENCY: Within-Group Annotation Trends Across Taxonomy Levels")
    print("=" * 120)
    print("\nNegative labeling delta = annotation quality degrades at higher specificity levels.")
    print("Positive failure delta = annotator failure rate worsens at higher specificity levels.")
    annotation_cross_cols = ['target', 'level_from', 'level_to', 'correct_labeling_rate_from', 'correct_labeling_rate_to', 'correct_labeling_rate_delta', 'annotator_failure_ratio_from', 'annotator_failure_ratio_to', 'annotator_failure_ratio_delta']
    print(cross_level_report[annotation_cross_cols].to_string(index=False))


In [ ]:
# Export disparity analysis results
if len(coverage_disparity_report) > 0:
    coverage_disparity_path = cfg.output_dir / 'coverage_disparity.tsv'
    coverage_disparity_report.to_csv(coverage_disparity_path, sep='\t', index=False)
    print(f"\nCoverage disparity analysis exported to: {coverage_disparity_path}")

if len(annotation_disparity_report) > 0:
    annotation_disparity_path = cfg.output_dir / 'annotation_disparity.tsv'
    annotation_disparity_report.to_csv(annotation_disparity_path, sep='\t', index=False)
    print(f"Annotation disparity analysis exported to: {annotation_disparity_path}")

if len(cross_level_report) > 0:
    cross_level_path = cfg.output_dir / 'cross_level_consistency.tsv'
    cross_level_report.to_csv(cross_level_path, sep='\t', index=False)
    print(f"Cross-level consistency analysis exported to: {cross_level_path}")

print(f"\nAll disparity analysis results exported to: {cfg.output_dir}")
